In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/kylin0338/themes2/themes_raw.csv.xlsx
/kaggle/input/datasets/kylin0338/aspect-sentiment-csv/aspect_sentiment.csv


In [2]:
!pip install -q sentence-transformers
import pandas as pd, numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer

pairs = pd.read_csv("/kaggle/input/datasets/kylin0338/aspect-sentiment-csv/aspect_sentiment.csv")
# /kaggle/input/datasets/kylin0338//aspect_sentiment.csv
neg = pairs[(pairs["aspect_sentiment"] == "negative") & (pairs["aspect_score"] > 0.7)].reset_index(drop=True)
print(neg["aspect"].value_counts())

model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")
emb = model.encode(neg["sentence"].tolist(), batch_size=256, show_progress_bar=True, normalize_embeddings=True)

aspect
battery             2418
network             2037
screen              1974
software            1797
customer_service    1567
camera              1141
sound                845
price                785
delivery             492
performance          456
design               430
usability            374
storage              348
build                324
Name: count, dtype: int64


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/59 [00:00<?, ?it/s]

In [3]:
K = 5
neg["cluster"] = -1
summary = []

for asp, g in neg.groupby("aspect"):
    if len(g) < 60:
        continue
    idx = g.index.values
    km = KMeans(n_clusters=K, random_state=42, n_init=10).fit(emb[idx])
    neg.loc[idx, "cluster"] = km.labels_

    for c in range(K):
        members = idx[km.labels_ == c]
        # representative sentences: closest to the cluster centre
        d = np.linalg.norm(emb[members] - km.cluster_centers_[c], axis=1)
        reps = neg.loc[members[np.argsort(d)[:3]], "sentence"].tolist()
        # keywords: top TF-IDF terms in the cluster
        tf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), max_features=2000)
        X = tf.fit_transform(neg.loc[members, "sentence"])
        top = np.array(tf.get_feature_names_out())[np.asarray(X.sum(axis=0)).ravel().argsort()[::-1][:6]]
        summary.append({"aspect": asp, "cluster": c, "size": len(members),
                        "keywords": ", ".join(top), "examples": " || ".join(s[:120] for s in reps)})

themes = pd.DataFrame(summary)
themes.to_csv("/kaggle/working/themes_raw.csv", index=False)
themes

,aspect,cluster,size,keywords,examples
0,battery,0,565,"battery, charge, hours, day, phone, long",The battery is defective. || the battery is po...
1,battery,1,502,"phone, battery, life, battery life, phone batt...",Battery life is poor but great phone. || Great...
2,battery,2,518,"battery, life, battery life, good, drains, great",Battery life is not great. || Battery life not...
3,battery,3,436,"charge, charger, charging, won, battery, hours",It still won't charge. || Tried another charge...
4,battery,4,397,"phone, charge, battery, charging, charger, cha...",Phone was working great the first couple of da...
...,...,...,...,...,...
65,usability,0,70,"fingerprint, phone, magnet, fingerprint magnet...",Phones had fingerprints on screen. || Fingerpr...
66,usability,1,185,"fingerprint, reader, fingerprint reader, senso...","The fingerprint reader is pretty much useless,..."
67,usability,2,22,"unlock, interface, face unlock, face, work, in...",Within 2 days face unlock simply stopped worki...
68,usability,3,37,"navigation, use, phone, google, button, app",Navigation bar doesn't go away unless you're p...


In [4]:
neg.to_csv("/kaggle/working/negative_with_clusters.csv", index=False)

In [5]:
# import pandas as pd

# # Read Excel file
# df = pd.read_excel("themes_raw.csv.xlsx")

# # Convert to CSV with UTF-8 encoding
# df.to_csv("themes_raw.csv", index=False, encoding="utf-8")

# print("Conversion successful!")

In [6]:
import os

os.listdir("/kaggle/working")

['negative_with_clusters.csv', 'themes_raw.csv', '__notebook__.ipynb']

In [7]:
import pandas as pd

df = pd.read_excel("/kaggle/input/datasets/kylin0338/themes2/themes_raw.csv.xlsx")


print(df.columns.tolist())

df.to_csv("/kaggle/working/themes_raw.csv",
          index=False,
          encoding="utf-8")

['aspect', 'cluster', 'size', 'keywords', 'examples', 'theme_label']


In [8]:
themes = pd.read_csv("/kaggle/working/themes_raw.csv")

print(themes.columns.tolist())
themes.head()

['aspect', 'cluster', 'size', 'keywords', 'examples', 'theme_label']


,aspect,cluster,size,keywords,examples,theme_label
0,battery,0,565,"battery, charge, hours, day, phone, long",The battery is defective. || the battery is po...,Battery: Defective or poor-performing battery
1,battery,1,502,"phone, battery, life, battery life, phone batt...",Battery life is poor but great phone. || Great...,Battery: Poor battery life
2,battery,2,518,"battery, life, battery life, good, drains, great",Battery life is not great. || Battery life not...,Battery: Poor battery life
3,battery,3,436,"charge, charger, charging, won, battery, hours",It still won't charge. || Tried another charge...,Battery: Charging failure
4,battery,4,397,"phone, charge, battery, charging, charger, cha...",Phone was working great the first couple of da...,Battery: Battery stops holding charge over time


In [9]:
# per product x aspect scores (only where there are enough mentions)
pairs["is_pos"] = (pairs["aspect_sentiment"] == "positive").astype(int)
pairs["is_neg"] = (pairs["aspect_sentiment"] == "negative").astype(int)

agg = pairs.groupby(["asin", "aspect"]).agg(
    mentions=("sentence", "count"),
    pos=("is_pos", "mean"),
    neg=("is_neg", "mean"),
).reset_index()
agg["net_score"] = agg["pos"] - agg["neg"]
agg = agg[agg["mentions"] >= 10]
agg.to_csv("/kaggle/working/product_aspect_scores.csv", index=False)